# Merging SI poplave data
Full, from feb 2025, just for Q100: http://www.evode.gov.si/index.php?id=119

## Setup


In [1]:
!which python
# !pip install nbformat pandas plotly requests
# !pip install nbformat --upgrade
# !pip install plotly

/Users/klemenkubelj/miniconda3/envs/cvar-masters/bin/python


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import geopandas as gpd
import pandas as pd

## Variables

In [11]:
GM = 0.5
GS = 1.5
GV = 2.5

## Define paths

In [5]:
# "/Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin/merged_v2/raw"
CLEANED_DATA_DIR = "/Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin/merged_v2/raw"

-------

# Processing

## Part 1: QGIS shapefiles
Shapefiles split by return period and intensity

In [8]:
# q10_dir = os.path.join(CLEANED_DATA_DIR, "q10")
q100_dir = os.path.join(CLEANED_DATA_DIR, "q100")
# q500_dir = os.path.join(HAND_CLEANED_DATA_DIR, "q500")
# q10_shapefiles = [os.path.join(q10_dir, f) for f in os.listdir(q10_dir) if f.endswith(".shp")]
q100_shapefiles = [os.path.join(q100_dir, f) for f in os.listdir(q100_dir) if f.endswith(".shp")]
# q500_shapefiles = [os.path.join(q500_dir, f) for f in os.listdir(q500_dir) if f.endswith(".shp")]
all_shapefiles = q100_shapefiles
print("Found ", len(all_shapefiles), " shapefiles")
# print("There are ", len(q10_shapefiles), " shapefiles for Q10")
print("There are ", len(q100_shapefiles), " shapefiles for Q100")
# print("There are ", len(q500_shapefiles), " shapefiles for Q500")

# Sanity check
# assert len(q10_shapefiles) == sum([i["q10"] for i in qgis_stats.values()])
# assert len(q100_shapefiles) == sum([i["q100"] for i in qgis_stats.values()])
# assert len(q500_shapefiles) == sum([i["q500"] for i in qgis_stats.values()])


Found  3  shapefiles
There are  3  shapefiles for Q100


In [12]:
def _get_globina(row):
    source = row["k_source_file"].lower()
    if source.endswith("gv.shp"):
        return GV
    elif source.endswith("gs.shp"):
        return GS
    elif source.endswith("gm.shp"):
        return GM
    raise ValueError(f"Unknown source: '{source}'")

def _clean_shapefile(_df, path):
    # Relative to DATA_DIR
    # RELDIR = "/Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin"
    source_file = os.path.relpath(path, CLEANED_DATA_DIR)
    _df["k_source_file"] = source_file
    # _df["k_region"] = source_file.split("/")[-1].split("_Q")[0] # Extract source from path
    _df["k_globina"] = _df.apply(lambda row: _get_globina(row), axis=1)
    _df["k_rp"] = _df.apply(lambda row: row["k_source_file"].split("/")[0], axis=1)
    # Replace all columns with lower case
    _df.columns = _df.columns.str.lower()
    return _df

def _read_shapefile(path):
    _df = gpd.read_file(path)
    # Check if _df is empty
    if _df.empty:
        print("Empty shapefile: ", path, ". Skipping...")
        return None
    _df = _clean_shapefile(_df, path)
    return _df

dfs = []
for path in all_shapefiles:
    _df = _read_shapefile(path)
    if _df is not None:
        dfs.append(_df)

print("Read ", len(dfs), " shapefiles")

Read  3  shapefiles


In [13]:
# Combine all dataframes into one
df_qgis_all = pd.concat(dfs)
df_qgis_all.shape

(304467, 26)

In [14]:
df_qgis_all.head()


,id_reg_gs,gs_id,gs_ime,verzija,datp_zac,datp_kon,razvoj,jezik,dat_zac,dat_kon,...,geometry,k_source_file,k_globina,k_rp,id_reg_gv,gv_id,gv_ime,id_reg_gm,gm_id,gm_ime
0,1390429.0,IKG100_GS_53103,"(območje) globin pri pretoku Q100: 0,5-1,5 m",v1-18,2019-09-23,None,New,slv,2007-07-21,None,...,"POLYGON ((605756.159 173512.254, 605755.494 17...",q100/IKG100_GS.shp,1.5,q100,NaN,NaN,NaN,NaN,NaN,NaN
1,1390430.0,IKG100_GS_53104,"(območje) globin pri pretoku Q100: 0,5-1,5 m",v1-18,2019-09-23,None,New,slv,2007-07-21,None,...,"POLYGON ((605075.629 173621.721, 605075.255 17...",q100/IKG100_GS.shp,1.5,q100,NaN,NaN,NaN,NaN,NaN,NaN
2,1390431.0,IKG100_GS_53105,"(območje) globin pri pretoku Q100: 0,5-1,5 m",v1-18,2019-09-23,None,New,slv,2007-07-21,None,...,"POLYGON ((604843.168 173656.631, 604843.878 17...",q100/IKG100_GS.shp,1.5,q100,NaN,NaN,NaN,NaN,NaN,NaN
3,1390442.0,IKG100_GS_52778,"(območje) globin pri pretoku Q100: 0,5-1,5 m",v1-18,2019-09-24,None,New,slv,2007-07-21,None,...,"POLYGON ((418565.202 79228.134, 418565.082 792...",q100/IKG100_GS.shp,1.5,q100,NaN,NaN,NaN,NaN,NaN,NaN
4,1390443.0,IKG100_GS_52779,"(območje) globin pri pretoku Q100: 0,5-1,5 m",v1-18,2019-09-24,None,New,slv,2007-07-21,None,...,"POLYGON ((418727.706 79266.885, 418727.829 792...",q100/IKG100_GS.shp,1.5,q100,NaN,NaN,NaN,NaN,NaN,NaN


In [80]:
# # To shapefile
# rad_q10 = '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Radeče/shape/Q10g_MERGED.shp'
# df_rad_q10.to_file(rad_q10, driver="ESRI Shapefile")


## Part 3: Merge QGIS and ArcGis data

In [16]:
print(sorted(df_qgis_all["k_globina"].unique()))
# print(sorted(df_argis_globine["k_globina"].unique()))
print("-"*100)
print(sorted(df_qgis_all["k_rp"].unique()))
# print(sorted(df_argis_globine["k_rp"].unique()))
print("-"*100)


[0.5, 1.5, 2.5]
----------------------------------------------------------------------------------------------------
['q100']
----------------------------------------------------------------------------------------------------


In [18]:
df_merged = df_qgis_all

# regions = sorted(df_merged["k_region"].unique())
# print("There are ", len(regions), " regions in merged data")


In [19]:
# Plot Q10
def plot(_df):
    return _df.explore(
        column="k_globina",
        scheme="naturalbreaks",  # use mapclassify's natural breaks scheme
        legend=True,  # show legend
        k=3,  # use 10 bins
        # tooltip=False,  # hide tooltip
        popup=["k_globina", "k_rp", "k_source_file", "rpn"],  # show popup (on-click)
        legend_kwds=dict(colorbar=False),  # do not use colorbar
        name="Globine",  # name of the layer in the map
    )

In [26]:
df_merged["k_globina"].value_counts()

k_globina
1.5    138751
0.5    126683
2.5     39033
Name: count, dtype: int64

In [27]:
# plot(df_merged[df_merged["k_rp"]=="q100"])

### Save merged data to new shapefile

In [28]:
OUT_DIR = "/Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin/merged_v2"
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)


In [96]:
# # Export to GeoJSON
# for rp in ["q10", "q100", "q500"]:
#     df_merged[df_merged["k_rp"]==rp].to_file(os.path.join(OUT_DIR, f"globine_{rp}.geojson"), driver="GeoJSON")


In [29]:
# Export to GPKG
for rp in ["q100"]:
    output_path = os.path.join(OUT_DIR, f"globine_{rp}.gpkg")
    
    # Remove existing file if it exists
    if os.path.exists(output_path):
        os.remove(output_path)
    df_merged[df_merged["k_rp"]==rp].to_file(output_path, driver="GPKG")